# CRISPR Gene Effect: creating categories

The more negative `GeneEffect` is, the more strongly gene knockout reduces cell-model survival or growth. A value near zero indicates no pronounced effect.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path("../../../data/CRISPRGeneEffect.csv").resolve()
crispr = pd.read_csv(DATA_PATH, low_memory=False).rename(columns={"Unnamed: 0": "ModelID"})

print("CRISPR table shape:", crispr.shape)
display(crispr.iloc[:3, :6])

## Inspecting the value range

First, inspect all available measurements. The plot limits the X-axis to `-3` through `1` so rare extreme values do not compress the main distribution. The values themselves are not removed from any calculations.


In [ ]:
effect_values = pd.Series(
    crispr.drop(columns="ModelID").to_numpy().ravel(),
    name="GeneEffect",
).dropna()

display(effect_values.describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(effect_values.clip(-3, 1), bins=100, color="steelblue", alpha=0.8)
ax.axvline(-1.0, color="darkred", linestyle="--", label="strong threshold = -1.0")
ax.axvline(-0.5, color="darkorange", linestyle="--", label="weak threshold = -0.5")
ax.axvline(0, color="black", linewidth=1, alpha=0.6)
ax.set_xlabel("CRISPR Gene Effect")
ax.set_ylabel("Number of measurements")
ax.set_title("CRISPR Gene Effect distribution")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## Three categories

We use fixed, interpretable thresholds:

- `no_effect`: `GeneEffect > -0.5` — no clear dependency is observed;
- `weak_effect`: `-1.0 < GeneEffect <= -0.5` — knockout moderately reduces survival or growth;
- `strong_effect`: `GeneEffect <= -1.0` — the cell strongly depends on the gene.

Fixed thresholds make genes comparable. Gene-specific tertiles would create three equally sized groups even for a gene whose knockout affects almost no models.


In [ ]:
def create_crispr_effect_categories(
    values,
    weak_threshold=-0.5,
    strong_threshold=-1.0,
):
    return pd.cut(
        values,
        bins=[-np.inf, strong_threshold, weak_threshold, np.inf],
        labels=["strong_effect", "weak_effect", "no_effect"],
        include_lowest=True,
    )

In [ ]:
effect_categories = create_crispr_effect_categories(effect_values)

category_summary = pd.DataFrame({
    "count": effect_categories.value_counts(sort=False),
    "percent": effect_categories.value_counts(normalize=True, sort=False).mul(100),
})
display(category_summary)

## Example for one gene

The same function can be applied to a table returned by `GeneOrchestrator`. KRAS is used below as an example.


In [ ]:
kras = (
    crispr[["ModelID", "KRAS (3845)"]]
    .dropna()
    .rename(columns={"KRAS (3845)": "GeneEffect"})
)
kras["EffectCategory"] = create_crispr_effect_categories(kras["GeneEffect"])

display(kras.head(10))
display(kras["EffectCategory"].value_counts(sort=False).to_frame("models"))

## Complete categorical ground-truth dataset

Now convert the full CRISPR matrix. Its structure remains unchanged: one row per `ModelID` and one column per gene. Each numerical Gene Effect value is replaced with its category. A categorical dtype stores repeated labels more efficiently than ordinary strings.


In [ ]:
def create_full_gt_dataset(
    crispr_df,
    weak_threshold=-0.5,
    strong_threshold=-1.0,
):
    gene_columns = crispr_df.columns.drop("ModelID")
    values = crispr_df[gene_columns].to_numpy()

    # Category codes: -1 = missing, 0 = strong, 1 = weak, 2 = no effect.
    category_codes = np.full(values.shape, -1, dtype=np.int8)
    category_codes[values <= strong_threshold] = 0
    category_codes[(values > strong_threshold) & (values <= weak_threshold)] = 1
    category_codes[values > weak_threshold] = 2

    labels = ["strong_effect", "weak_effect", "no_effect"]
    categorical_columns = {
        gene: pd.Categorical.from_codes(category_codes[:, index], categories=labels)
        for index, gene in enumerate(gene_columns)
    }

    gt = pd.DataFrame(categorical_columns)
    gt.insert(0, "ModelID", crispr_df["ModelID"].to_numpy())
    return gt


full_gt = create_full_gt_dataset(crispr)

print("Original CRISPR table shape:", crispr.shape)
print("Full ground-truth shape:", full_gt.shape)
display(full_gt.iloc[:5, :8])

## Removing uninformative genes and genes with many missing values

A gene is retained only when both conditions hold:

1. At least `MIN_EFFECT_MODELS` models have `weak_effect` or `strong_effect`.
2. The fraction of missing measurements does not exceed `MAX_MISSING_FRACTION`.

By default, at most 20% missing values are allowed, so measurements must be available for at least 80% of the 1,208 models. Both thresholds can be changed in the next cell.


In [ ]:
MIN_EFFECT_MODELS = 1
MAX_MISSING_FRACTION = 0.20

gene_columns = full_gt.columns.drop("ModelID")
n_models = len(full_gt)

gene_category_summary = pd.DataFrame({
    "strong_effect": full_gt[gene_columns].eq("strong_effect").sum(),
    "weak_effect": full_gt[gene_columns].eq("weak_effect").sum(),
    "no_effect": full_gt[gene_columns].eq("no_effect").sum(),
    "missing": full_gt[gene_columns].isna().sum(),
})
gene_category_summary["measured"] = n_models - gene_category_summary["missing"]
gene_category_summary["missing_fraction"] = (
    gene_category_summary["missing"] / n_models
)
gene_category_summary["weak_or_strong"] = (
    gene_category_summary["weak_effect"]
    + gene_category_summary["strong_effect"]
)

has_effect = gene_category_summary["weak_or_strong"] >= MIN_EFFECT_MODELS
has_enough_measurements = (
    gene_category_summary["missing_fraction"] <= MAX_MISSING_FRACTION
)
genes_to_keep = gene_category_summary.index[
    has_effect & has_enough_measurements
].tolist()

filtered_gt = full_gt[["ModelID", *genes_to_keep]].copy()

print("Genes before filtering:", len(gene_columns))
print("Genes removed without weak/strong effect:", int((~has_effect).sum()))
print(
    "Genes removed with a missing fraction above",
    f"{MAX_MISSING_FRACTION:.0%}:",
    int((has_effect & ~has_enough_measurements).sum()),
)
print("Genes after both filters:", len(genes_to_keep))
print("Cleaned ground-truth shape:", filtered_gt.shape)

display(
    gene_category_summary.loc[genes_to_keep]
    .sort_values("weak_or_strong", ascending=False)
    .head(10)
)


## How often one effect dominates

For each gene, count how many categories are observed and identify the category with the largest share among non-missing measurements. This helps distinguish universally essential genes from context-dependent genes.


In [ ]:
category_columns = ["strong_effect", "weak_effect", "no_effect"]
category_fractions = gene_category_summary[category_columns].div(
    gene_category_summary["measured"].replace(0, np.nan),
    axis=0,
)
gene_category_summary["observed_categories"] = (
    gene_category_summary[category_columns] > 0
).sum(axis=1)
gene_category_summary["dominant_category"] = category_fractions.idxmax(axis=1)
gene_category_summary["dominant_fraction"] = category_fractions.max(axis=1)

display(
    gene_category_summary["observed_categories"]
    .value_counts()
    .sort_index()
    .rename_axis("Number of observed categories")
    .to_frame("Genes")
)
display(
    gene_category_summary["dominant_category"]
    .value_counts()
    .to_frame("Genes by dominant category")
)
print("Genes with strong_effect only:", int((category_fractions["strong_effect"] == 1).sum()))
print("Genes with no_effect only:", int((category_fractions["no_effect"] == 1).sum()))


## Genes represented in all three categories

Collect genes for which `no_effect`, `weak_effect`, and `strong_effect` all occur among the cell models. These genes are especially useful for studying context dependence because their knockout effect varies across models.

Save the complete gene list with category statistics. Apply the missing-value threshold defined above to the working matrix as an additional filter.


In [ ]:
has_all_three_categories = (
    gene_category_summary[["strong_effect", "weak_effect", "no_effect"]] > 0
).all(axis=1)

all_three_genes = gene_category_summary.index[has_all_three_categories].tolist()
clean_all_three_genes = gene_category_summary.index[
    has_all_three_categories & has_enough_measurements
].tolist()

all_three_gene_summary = (
    gene_category_summary.loc[all_three_genes]
    .sort_values(["strong_effect", "weak_effect"], ascending=False)
    .reset_index(names="Gene")
)
all_three_categories_gt = full_gt[["ModelID", *all_three_genes]].copy()
clean_all_three_categories_gt = full_gt[["ModelID", *clean_all_three_genes]].copy()

print("Genes with all three categories:", len(all_three_genes))
print(
    "After the missing-value filter:",
    len(clean_all_three_genes),
)
print(
    "Removed because of missing values:",
    len(all_three_genes) - len(clean_all_three_genes),
)
display(all_three_gene_summary.head(10))

ALL_THREE_GENES_PATH = DATA_PATH.parent / "genes_with_all_three_categories.csv"
ALL_THREE_GT_PATH = DATA_PATH.parent / "CRISPRGeneEffect_GT_all_three_categories.csv"

all_three_gene_summary.to_csv(ALL_THREE_GENES_PATH, index=False)
clean_all_three_categories_gt.to_csv(ALL_THREE_GT_PATH, index=False)

print("Gene list saved to:", ALL_THREE_GENES_PATH)
print("Working ground truth saved to:", ALL_THREE_GT_PATH)


In [ ]:
OUTPUT_PATH = DATA_PATH.parent / "CRISPRGeneEffect_GT.csv"
filtered_gt.to_csv(OUTPUT_PATH, index=False)
print("Cleaned ground truth saved to:", OUTPUT_PATH)


### Retrieving ground truth for one gene

The target variable for a specific gene can be extracted quickly from the cleaned ground-truth dataset.


In [ ]:
def get_gene_gt(gt_df, gene):
    return (
        gt_df[["ModelID", gene]]
        .dropna()
        .rename(columns={gene: "EffectCategory"})
        .reset_index(drop=True)
    )

kras_gt = get_gene_gt(filtered_gt, "A1BG (1)")
display(kras_gt.head(10))
display(kras_gt["EffectCategory"].value_counts(sort=False).to_frame("models"))


## Possible fourth category

Large positive values such as `GeneEffect > 0.5` could later be separated into a `growth_advantage` category, indicating that the model grows better after knockout. This category is omitted for now because the current objective is the strength of reduced survival and positive values are rare.
